# Agent Middleware & Summarizer in LangGraph / LangChain

This notebook demonstrates:
1. **Thread Memory Checkpointer**: Storing thread state across messages.
2. **Summarizer Middleware**: Automatically summarizing older context when message history grows long to keep memory efficient.

### Step 1: Imports & Setup

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, RemoveMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# Initialize ChatGroq model
model = ChatGroq(model="llama-3.3-70b-versatile")

### Part 1: Basic Thread Memory Checkpointer

In [2]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [add, multiply]
memory = MemorySaver()
agent = create_react_agent(model, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "1"}}
questions = ["What is 2+2?", "What is 4*4?"]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"--- Question: {q} ---")
    print(f"Response: {response['messages'][-1].content}")
    print(f"Total Messages: {len(response['messages'])}\n")

--- Question: What is 2+2? ---
Response: The answer is 4.
Total Messages: 4

--- Question: What is 4*4? ---
Response: The answer is 16.
Total Messages: 8


### Part 2: Summarizer Middleware
Automatically triggers context summarization and prunes older raw messages when history length exceeds a threshold.

In [3]:
# Define state schema extending MessagesState with a summary field
class State(MessagesState):
    summary: str

# 1. Model Node
def call_model(state: State):
    summary = state.get("summary", "")
    if summary:
        system_message = f"Summary of conversation so far: {summary}"
        messages = [SystemMessage(content=system_message)] + state["messages"]
    else:
        messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": response}

# 2. Summarizer Middleware Node
def summarize_conversation(state: State):
    summary = state.get("summary", "")
    if summary:
        summary_prompt = (
            f"This is summary of the conversation so far: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
    else:
        summary_prompt = "Create a summary of the conversation above:"
    
    messages = state["messages"] + [HumanMessage(content=summary_prompt)]
    response = model.invoke(messages)
    
    # Remove older messages to keep message history lightweight
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

# 3. Conditional Edge router
def should_continue(state: State):
    messages = state["messages"]
    if len(messages) > 4:
        return "summarize_conversation"
    return END

# Build workflow
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node("summarize_conversation", summarize_conversation)

workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue, ["summarize_conversation", END])
workflow.add_edge("summarize_conversation", END)

# Compile graph with checkpointer
memory_summarizer = MemorySaver()
app = workflow.compile(checkpointer=memory_summarizer)

# Execute multi-turn conversation
config = {"configurable": {"thread_id": "summary_thread_1"}}

r1 = app.invoke({"messages": [HumanMessage(content="Hi, I am Alice and I love machine learning.")]}, config)
print(f"Turn 1 Response: {r1['messages'][-1].content[:80]}...")

r2 = app.invoke({"messages": [HumanMessage(content="What is my name and favorite topic?")]}, config)
print(f"Turn 2 Response: {r2['messages'][-1].content}")

r3 = app.invoke({"messages": [HumanMessage(content="Can you remind me what we talked about so far?")]}, config)
print(f"Turn 3 Response: {r3['messages'][-1].content[:140]}...")

print("\n--- Current Conversation Summary in State ---")
print(app.get_state(config).values.get("summary"))

Turn 1 Response: Hello Alice! It's great to meet you. Machine learning is a fascinating field...
Turn 2 Response: Your name is Alice, and your favorite topic is machine learning.
Turn 3 Response: We've had a brief conversation so far. Here's a summary: You introduced yourself as Alice and mentioned that you love machine learning...

--- Current Conversation Summary in State ---
Here's a summary of our conversation: Alice introduced herself, sharing her interest in machine learning. We discussed her interests and I provided a recap of our conversation when asked.
